In [1]:

import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/adarshsng/lending-club-loan-data-csv/loan.csv
/kaggle/input/datasets/adarshsng/lending-club-loan-data-csv/LCDataDictionary.xlsx


In [2]:
import pandas as pd
import os

# Try local paths first, then Kaggle path
local_paths = [
    'archive/loan1.csv',
    'back up data/archive/loan.csv',
    'archive/loan.csv'
]
csv_path = None
for path in local_paths:
    if os.path.exists(path):
        csv_path = path
        break

if csv_path is None:
    csv_path = '/kaggle/input/datasets/adarshsng/lending-club-loan-data-csv/loan.csv'

print(f"Loading dataset from: {csv_path}")
df = pd.read_csv(csv_path, low_memory=False)
print(df.shape)          # Should be (2260668, 145)
print(df['loan_status'].value_counts())

(2260668, 145)
loan_status
Fully Paid                                             1041952
Current                                                 919695
Charged Off                                             261655
Late (31-120 days)                                       21897
In Grace Period                                           8952
Late (16-30 days)                                         3737
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     31
Name: count, dtype: int64


In [3]:
# STEP 1: Map target variable → binary, drop ambiguous rows

target_map = {
    'Fully Paid': 0,
    'Current': 0,
    'Does not meet the credit policy. Status:Fully Paid': 0,
    'Charged Off': 1,
    'Default': 1,
    'Does not meet the credit policy. Status:Charged Off': 1,
}


df['default'] = df['loan_status'].map(target_map)

df = df.dropna(subset=['default'])
df['default'] = df['default'].astype(int)

# Sanity check
print(f"Rows after filtering: {df.shape[0]:,}")
print(f"\nDefault rate: {df['default'].mean():.2%}")
print(f"\nValue counts:\n{df['default'].value_counts()}")

Rows after filtering: 2,226,082

Default rate: 11.79%

Value counts:
default
0    1963635
1     262447
Name: count, dtype: int64


In [4]:
# STEP 2: Select 15 power features

features = [
    'loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade',
    'emp_length', 'home_ownership', 'annual_inc',
    'verification_status', 'purpose', 'dti',
    'delinq_2yrs', 'inq_last_6mths', 'revol_util', 'pub_rec'
]

df = df[features + ['default']]

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nMissing values:\n{df.isnull().sum()}")

Shape: (2226082, 16)

Columns: ['loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'purpose', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'revol_util', 'pub_rec', 'default']

Missing values:
loan_amnt                   0
term                        0
int_rate                    0
grade                       0
sub_grade                   0
emp_length             143970
home_ownership              0
annual_inc                  4
verification_status         0
purpose                     0
dti                      1669
delinq_2yrs                29
inq_last_6mths             30
revol_util               1757
pub_rec                    29
default                     0
dtype: int64


In [5]:
# STEP 3: Clean features

if df['int_rate'].dtype == object:
    df['int_rate'] = df['int_rate'].str.replace('%', '').str.strip().astype(float)

if df['revol_util'].dtype == object:
    df['revol_util'] = df['revol_util'].str.replace('%', '').str.strip().astype(float)

emp_length_map = {
    '< 1 year': 0,
    '1 year': 1,
    '2 years': 2,
    '3 years': 3,
    '4 years': 4,
    '5 years': 5,
    '6 years': 6,
    '7 years': 7,
    '8 years': 8,
    '9 years': 9,
    '10+ years': 10
}
df['emp_length'] = df['emp_length'].map(emp_length_map)

grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
df['grade'] = df['grade'].map(grade_map)

if df['term'].dtype == object:
    df['term'] = df['term'].str.strip().str.extract('(\d+)').astype(int)

print("Dtypes after cleaning:")
print(df.dtypes)
print(f"\nSample:\n{df.head(3)}")

<>:28: SyntaxWarning: invalid escape sequence '\d'
<>:28: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_55/1248585558.py:28: SyntaxWarning: invalid escape sequence '\d'
  df['term'] = df['term'].str.strip().str.extract('(\d+)').astype(int)


Dtypes after cleaning:
loan_amnt                int64
term                     int64
int_rate               float64
grade                    int64
sub_grade               object
emp_length             float64
home_ownership          object
annual_inc             float64
verification_status     object
purpose                 object
dti                    float64
delinq_2yrs            float64
inq_last_6mths         float64
revol_util             float64
pub_rec                float64
default                  int64
dtype: object

Sample:
   loan_amnt  term  int_rate  grade sub_grade  emp_length home_ownership  \
0       2500    36     13.56      3        C1        10.0           RENT   
1      30000    60     18.94      4        D2        10.0       MORTGAGE   
2       5000    36     17.97      4        D1         6.0       MORTGAGE   

   annual_inc verification_status             purpose    dti  delinq_2yrs  \
0     55000.0        Not Verified  debt_consolidation  18.24          0.0   

In [ ]:
# STEP 4: Label encode categoricals

from sklearn.preprocessing import LabelEncoder

cat_cols = ['sub_grade', 'home_ownership', 'verification_status', 'purpose']

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

print("Dtypes after encoding:")
print(df.dtypes)
print(f"\nSample:\n{df[cat_cols].head(3)}")

In [ ]:
# STEP 5: Feature Engineering

df['income_to_loan_ratio'] = df['annual_inc'] / (df['loan_amnt'] + 1)
df['interest_burden'] = df['int_rate'] * df['loan_amnt'] / 100

print(f"Shape: {df.shape}")
print(f"\nNew features sample:\n{df[['income_to_loan_ratio', 'interest_burden']].head(3)}")

In [ ]:
# STEP 6: Handle missing values

# Fill numerics with median
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

print("Missing values after fill:")
print(df.isnull().sum())
print(f"\nFinal shape: {df.shape}")

In [ ]:
# TRAIN / TEST SPLIT + LIGHTGBM

import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

X = df.drop('default', axis=1)
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train,
          eval_set=[(X_test, y_test)],
          callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)])

y_pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"\nAUC: {auc:.4f}")
print(classification_report(y_test, (y_pred_proba > 0.5).astype(int)))

In [ ]:
model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.01,
    num_leaves=64,
    min_child_samples=100,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train,
          eval_set=[(X_test, y_test)],
          callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])

y_pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"\nAUC: {auc:.4f}")
print(classification_report(y_test, (y_pred_proba > 0.5).astype(int)))

In [ ]:
model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.01,
    num_leaves=64,
    min_child_samples=100,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train,
          eval_set=[(X_test, y_test)],
          eval_metric='auc',
          callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])

y_pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)

# Lower threshold since defaults are minority class
threshold = 0.3
y_pred = (y_pred_proba > threshold).astype(int)

print(f"\nAUC: {auc:.4f}")
print(f"Threshold: {threshold}")
print(classification_report(y_test, y_pred))

In [ ]:
import numpy as np

print("Probability stats:")
print(f"Min: {y_pred_proba.min():.4f}")
print(f"Max: {y_pred_proba.max():.4f}")
print(f"Mean: {y_pred_proba.mean():.4f}")
print(f"Median: {np.median(y_pred_proba):.4f}")

print(f"\nHow many above 0.3: {(y_pred_proba > 0.3).sum()}")
print(f"How many above 0.2: {(y_pred_proba > 0.2).sum()}")
print(f"How many above 0.15: {(y_pred_proba > 0.15).sum()}")
print(f"How many above 0.13: {(y_pred_proba > 0.13).sum()}")

In [ ]:
# Retrain without early stopping, more trees
model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    min_child_samples=50,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)

print("Probability stats:")
print(f"Min: {y_pred_proba.min():.4f}")
print(f"Max: {y_pred_proba.max():.4f}")
print(f"Mean: {y_pred_proba.mean():.4f}")

threshold = 0.3
y_pred = (y_pred_proba > threshold).astype(int)
print(f"\nAUC: {auc:.4f}")
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score
import matplotlib.pyplot as plt

precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_threshold = thresholds[f1_scores.argmax()]

print(f"Best threshold: {best_threshold:.4f}")
print(f"Best F1: {f1_scores.max():.4f}")

y_pred_best = (y_pred_proba > best_threshold).astype(int)
print(f"\nFinal Classification Report:")
print(classification_report(y_test, y_pred_best))

plt.plot(thresholds, f1_scores[:-1])
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.title('F1 Score vs Threshold')
plt.axvline(best_threshold, color='red', linestyle='--', label=f'Best: {best_threshold:.2f}')
plt.legend()
plt.show()

**Low / Medium / High:**

In [ ]:
import numpy as np

# Risk bucketing based on probability of default
def assign_risk(prob):
    if prob < 0.3:
        return 'Low'
    elif prob < 0.6:
        return 'Medium'
    else:
        return 'High'

risk_df = X_test.copy()
risk_df['default_actual'] = y_test.values
risk_df['pd_score'] = y_pred_proba
risk_df['risk_tier'] = risk_df['pd_score'].apply(assign_risk)

print("Risk Tier Distribution:")
print(risk_df['risk_tier'].value_counts())
print(f"\nDefault rate per tier:")
print(risk_df.groupby('risk_tier')['default_actual'].mean().sort_values(ascending=False))

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
sample = X_test.sample(5000, random_state=42)
shap_values = explainer.shap_values(sample)

shap.summary_plot(shap_values, sample, plot_type="bar", show=True)

shap.summary_plot(shap_values, sample, show=True)

In [ ]:
import os
os.makedirs('plots', exist_ok=True)
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Loan Default — EDA Visualizations', fontsize=16, fontweight='bold')

# Plot 1 — Default rate by grade
grade_default = df.groupby('grade')['default'].mean().reset_index()
axes[0,0].bar(grade_default['grade'], grade_default['default'], color='steelblue')
axes[0,0].set_title('Default Rate by Grade')
axes[0,0].set_xlabel('Grade (1=A, 7=G)')
axes[0,0].set_ylabel('Default Rate')

# Plot 2 — PD score distribution
axes[0,1].hist(y_pred_proba, bins=50, color='coral', edgecolor='black')
axes[0,1].set_title('Probability of Default Distribution')
axes[0,1].set_xlabel('PD Score')
axes[0,1].set_ylabel('Count')

# Plot 3 — Risk tier distribution
tier_counts = risk_df['risk_tier'].value_counts()
axes[0,2].bar(tier_counts.index, tier_counts.values, color=['green','orange','red'])
axes[0,2].set_title('Risk Tier Distribution')
axes[0,2].set_xlabel('Risk Tier')
axes[0,2].set_ylabel('Borrower Count')

# Plot 4 — Default rate by risk tier
tier_default = risk_df.groupby('risk_tier')['default_actual'].mean()
axes[1,0].bar(tier_default.index, tier_default.values, color=['green','orange','red'])
axes[1,0].set_title('Actual Default Rate by Risk Tier')
axes[1,0].set_xlabel('Risk Tier')
axes[1,0].set_ylabel('Default Rate')

# Plot 5 — int_rate vs default
axes[1,1].boxplot([df[df['default']==0]['int_rate'], df[df['default']==1]['int_rate']],
                   labels=['Not Defaulted', 'Defaulted'])
axes[1,1].set_title('Interest Rate vs Default')
axes[1,1].set_ylabel('Interest Rate')

# Plot 6 — DTI vs default
axes[1,2].boxplot([df[df['default']==0]['dti'], df[df['default']==1]['dti']],
                   labels=['Not Defaulted', 'Defaulted'])
axes[1,2].set_title('DTI vs Default')
axes[1,2].set_ylabel('DTI')

plt.tight_layout()
plt.savefig('plots/eda_visualizations.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import os
os.makedirs('plots', exist_ok=True)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── Plot 1: Default Rate by Grade ──────────────────────────────
fig, ax = plt.subplots(figsize=(8,5))
grade_default = df.groupby('grade')['default'].mean().reset_index()
ax.bar(grade_default['grade'], grade_default['default'], color='steelblue')
ax.set_title('Default Rate by Grade')
ax.set_xlabel('Grade (1=A → 7=G)')
ax.set_ylabel('Default Rate')
plt.tight_layout()
plt.savefig('plots/plot1_grade_default.png', dpi=150)
plt.show()

# ── Plot 2: PD Score Distribution ──────────────────────────────
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(y_pred_proba, bins=50, color='coral', edgecolor='black')
ax.axvline(0.3, color='orange', linestyle='--', label='Medium threshold (0.3)')
ax.axvline(0.6, color='red', linestyle='--', label='High threshold (0.6)')
ax.set_title('Probability of Default Distribution')
ax.set_xlabel('PD Score')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('plots/plot2_pd_distribution.png', dpi=150)
plt.show()

# ── Plot 3: Risk Tier Distribution ─────────────────────────────
fig, ax = plt.subplots(figsize=(8,5))
tier_counts = risk_df['risk_tier'].value_counts().reindex(['Low','Medium','High'])
ax.bar(tier_counts.index, tier_counts.values, color=['green','orange','red'])
for i, v in enumerate(tier_counts.values):
    ax.text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')
ax.set_title('Risk Tier Distribution')
ax.set_xlabel('Risk Tier')
ax.set_ylabel('Borrower Count')
plt.tight_layout()
plt.savefig('plots/plot3_risk_tiers.png', dpi=150)
plt.show()

# ── Plot 4: Default Rate by Risk Tier ──────────────────────────
fig, ax = plt.subplots(figsize=(8,5))
tier_default = risk_df.groupby('risk_tier')['default_actual'].mean().reindex(['Low','Medium','High'])
bars = ax.bar(tier_default.index, tier_default.values, color=['green','orange','red'])
for bar, val in zip(bars, tier_default.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.1%}', ha='center', fontweight='bold')
ax.set_title('Actual Default Rate by Risk Tier')
ax.set_xlabel('Risk Tier')
ax.set_ylabel('Default Rate')
plt.tight_layout()
plt.savefig('plots/plot4_default_by_tier.png', dpi=150)
plt.show()

# ── Plot 5: Correlation Heatmap ────────────────────────────────
fig, ax = plt.subplots(figsize=(12,10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('plots/plot5_correlation.png', dpi=150)
plt.show()

# ── Plot 6: Interest Rate vs Default (Boxplot) ─────────────────
fig, ax = plt.subplots(figsize=(8,5))
df.boxplot(column='int_rate', by='default', ax=ax,
           patch_artist=True)
ax.set_title('Interest Rate vs Default')
ax.set_xlabel('Default (0=No, 1=Yes)')
ax.set_ylabel('Interest Rate')
plt.suptitle('')
plt.tight_layout()
plt.savefig('plots/plot6_intrate_default.png', dpi=150)
plt.show()

# ── Plot 7: DTI vs Default (Boxplot) ───────────────────────────
fig, ax = plt.subplots(figsize=(8,5))
df.boxplot(column='dti', by='default', ax=ax)
ax.set_title('DTI vs Default')
ax.set_xlabel('Default (0=No, 1=Yes)')
ax.set_ylabel('Debt-to-Income Ratio')
plt.suptitle('')
plt.tight_layout()
plt.savefig('plots/plot7_dti_default.png', dpi=150)
plt.show()

# ── Plot 8: Loan Amount Distribution by Default ────────────────
fig, ax = plt.subplots(figsize=(8,5))
df[df['default']==0]['loan_amnt'].hist(bins=50, alpha=0.6, label='Not Defaulted', ax=ax, color='steelblue')
df[df['default']==1]['loan_amnt'].hist(bins=50, alpha=0.6, label='Defaulted', ax=ax, color='red')
ax.set_title('Loan Amount Distribution by Default')
ax.set_xlabel('Loan Amount')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('plots/plot8_loanamnt_default.png', dpi=150)
plt.show()

# ── Plot 9: Default Rate by Employment Length ──────────────────
fig, ax = plt.subplots(figsize=(8,5))
emp_default = df.groupby('emp_length')['default'].mean().reset_index()
ax.bar(emp_default['emp_length'], emp_default['default'], color='purple', alpha=0.7)
ax.set_title('Default Rate by Employment Length')
ax.set_xlabel('Employment Length (years)')
ax.set_ylabel('Default Rate')
plt.tight_layout()
plt.savefig('plots/plot9_emplength_default.png', dpi=150)
plt.show()

# ── Plot 10: ROC Curve ─────────────────────────────────────────
from sklearn.metrics import roc_curve
fig, ax = plt.subplots(figsize=(8,5))
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = 0.7686)')
ax.plot([0,1], [0,1], color='navy', linestyle='--')
ax.set_title('ROC Curve')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()
plt.tight_layout()
plt.savefig('plots/plot10_roc.png', dpi=150)
plt.show()

print("All 10 plots saved ✅")

In [ ]:
# STEP 7: Save production artifacts
import pickle
import os

# Create artifacts directory if not exists
os.makedirs('artifacts', exist_ok=True)

# Save LightGBM model
with open('artifacts/lgbm_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save dict of LabelEncoders
with open('artifacts/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

# Save threshold (0.61)
with open('artifacts/threshold.pkl', 'wb') as f:
    pickle.dump(0.61, f)

# Save ordered feature columns list
feature_cols = X.columns.tolist()
with open('artifacts/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print("Artifacts saved successfully:")
print(" - artifacts/lgbm_model.pkl")
print(" - artifacts/label_encoders.pkl")
print(" - artifacts/threshold.pkl")
print(" - artifacts/feature_columns.pkl")

In [ ]:
# STEP 8: Verify loaded artifacts
import pickle
import pandas as pd
import numpy as np

# Load artifacts back
with open('artifacts/lgbm_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

with open('artifacts/label_encoders.pkl', 'rb') as f:
    loaded_encoders = pickle.load(f)

with open('artifacts/threshold.pkl', 'rb') as f:
    loaded_threshold = pickle.load(f)

with open('artifacts/feature_columns.pkl', 'rb') as f:
    loaded_features = pickle.load(f)

print("Artifacts loaded successfully!")
print(f"Loaded threshold: {loaded_threshold}")
print(f"Loaded feature columns: {loaded_features}")

# Run a test prediction using the first row of X_test
if 'X_test' in globals():
    sample_row = X_test.iloc[[0]].copy()
else:
    # Fallback to a dummy row with correct columns if run in isolation
    sample_row = pd.DataFrame(np.zeros((1, len(loaded_features))), columns=loaded_features)

print("\nSample input row:")
print(sample_row)

pred_proba = loaded_model.predict_proba(sample_row)[:, 1][0]
pred_class = int(pred_proba > loaded_threshold)

print(f"\nPrediction probability: {pred_proba:.4f}")
print(f"Prediction class (threshold={loaded_threshold}): {pred_class}")
print("Verification successful!")